# XMM MOS1/MOS2/PN spectral fitting with XSPEC (no XGA dependency)

Standalone notebook that generates an XSPEC `.xcm` script, runs XSPEC from command line, saves best-fit parameters, and plots spectra.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import shutil
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass
class SpectrumConfig:
    name: str
    pha: Path
    bkg: Path
    rmf: Path
    arf: Path


@dataclass
class ThreeSpectrumSetup:
    m1: SpectrumConfig
    m2: SpectrumConfig
    pn: SpectrumConfig

    @property
    def spectra(self):
        return [self.m1, self.m2, self.pn]


def validate_setup(setup: ThreeSpectrumSetup) -> None:
    missing = []
    for sp in setup.spectra:
        for label, p in {"pha": sp.pha, "bkg": sp.bkg, "rmf": sp.rmf, "arf": sp.arf}.items():
            if not Path(p).exists():
                missing.append(f"{sp.name}:{label}:{p}")
    if missing:
        raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))


In [ ]:
setup = ThreeSpectrumSetup(
    m1=SpectrumConfig("m1", Path("/path/to/m1_src.pha"), Path("/path/to/m1_bkg.pha"), Path("/path/to/m1.rmf"), Path("/path/to/m1.arf")),
    m2=SpectrumConfig("m2", Path("/path/to/m2_src.pha"), Path("/path/to/m2_bkg.pha"), Path("/path/to/m2.rmf"), Path("/path/to/m2.arf")),
    pn=SpectrumConfig("pn", Path("/path/to/pn_src.pha"), Path("/path/to/pn_bkg.pha"), Path("/path/to/pn.rmf"), Path("/path/to/pn.arf")),
)

redshift = 0.20
nh_1e22 = 0.03
output_prefix = Path("./fit_outputs/xmm_cluster")
output_prefix.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
def build_three_spec_xcm(
    setup: ThreeSpectrumSetup,
    output_prefix: Path,
    redshift: float,
    nh_1e22: float,
    kT_init_keV: float = 5.0,
    abundance_init: float = 0.3,
    norm_init: float = 1e-3,
    e_low_keV: float = 0.3,
    e_high_keV: float = 10.0,
    err_delta: float = 1.0,
) -> Path:
    xcm_path = output_prefix.with_suffix(".xcm")
    fit_csv = output_prefix.with_name(output_prefix.name + "_fit_results.csv")

    lines = [
        "autosave off",
        "query yes",
        "setplot energy",
        "statistic cstat",
        "abund angr",
    ]

    for i, sp in enumerate(setup.spectra, start=1):
        lines.extend([
            f'data {i}:1 "{sp.pha}"',
            f'backgrnd {i} "{sp.bkg}"',
            f'response {i} "{sp.rmf}"',
            f'arf {i} "{sp.arf}"',
            f'ignore {i}:**-{e_low_keV} {e_high_keV}-**',
        ])

    lines.extend([
        "model constant*tbabs*apec",
        "/*",
        "newpar 1 1.0",
        "freeze 1",
        f"newpar 2 {nh_1e22}",
        "freeze 2",
        f"newpar 3 {kT_init_keV}",
        f"newpar 4 {abundance_init}",
        f"newpar 5 {redshift}",
        "freeze 5",
        f"newpar 6 {norm_init}",
        "newpar 7 1.0",
        "thaw 7",
        "newpar 8 1.0",
        "thaw 8",
        "fit 300",
        f"error {err_delta} 3 4 6",
        "plot ldata del",
        f'set fsum [open "{fit_csv}" w]',
        'puts $fsum "kT_keV,kT_err_lo,kT_err_hi,abundance_solar,abund_err_lo,abund_err_hi,norm,norm_err_lo,norm_err_hi,cstat,dof,redshift,nh_1e22"',
        'tclout param 3',
        'set kt [lindex $xspec_tclout 0]',
        'tclout error 3',
        'set kt_lo [lindex $xspec_tclout 0]',
        'set kt_hi [lindex $xspec_tclout 1]',
        'tclout param 4',
        'set ab [lindex $xspec_tclout 0]',
        'tclout error 4',
        'set ab_lo [lindex $xspec_tclout 0]',
        'set ab_hi [lindex $xspec_tclout 1]',
        'tclout param 6',
        'set nm [lindex $xspec_tclout 0]',
        'tclout error 6',
        'set nm_lo [lindex $xspec_tclout 0]',
        'set nm_hi [lindex $xspec_tclout 1]',
        'tclout stat',
        'set cstat [lindex $xspec_tclout 0]',
        'tclout dof',
        'set dof [lindex $xspec_tclout 0]',
        f'puts $fsum "$kt,$kt_lo,$kt_hi,$ab,$ab_lo,$ab_hi,$nm,$nm_lo,$nm_hi,$cstat,$dof,{redshift},{nh_1e22}"',
        'close $fsum',
    ])

    for i, sp in enumerate(setup.spectra, start=1):
        spec_csv = output_prefix.with_name(output_prefix.name + f"_{sp.name}_plot.csv")
        lines.extend([
            f'set fp{i} [open "{spec_csv}" w]',
            f'set x{i} [tcloutr plot data x {i}]',
            f'set y{i} [tcloutr plot data y {i}]',
            f'set xe{i} [tcloutr plot data xerr {i}]',
            f'set ye{i} [tcloutr plot data yerr {i}]',
            f'set ym{i} [tcloutr plot data model {i}]',
            f'puts $fp{i} "energy_keV,rate,energy_err,rate_err,model_rate"',
            f'set n{i} [llength $x{i}]',
            f'for {{set j 0}} {{$j < $n{i}}} {{incr j}} {{',
            f'  set row "[lindex $x{i} $j],[lindex $y{i} $j],[lindex $xe{i} $j],[lindex $ye{i} $j],[lindex $ym{i} $j]"',
            f'  puts $fp{i} $row',
            '}',
            f'close $fp{i}',
        ])

    lines.append('exit')
    xcm_path.write_text("\n".join(lines) + "\n")
    return xcm_path


In [ ]:
def run_xspec(xcm_path: Path, xspec_cmd: str = "xspec") -> subprocess.CompletedProcess:
    if shutil.which(xspec_cmd) is None:
        raise RuntimeError(f"'{xspec_cmd}' not found in PATH. Load HEASOFT/XSPEC first.")

    cmd = [xspec_cmd, "-", str(xcm_path)]
    proc = subprocess.run(cmd, capture_output=True, text=True)

    log_path = xcm_path.with_suffix(".log")
    log_path.write_text(proc.stdout + "\n\nSTDERR:\n" + proc.stderr)

    if proc.returncode != 0:
        raise RuntimeError(f"XSPEC failed with code {proc.returncode}. Check {log_path}")
    return proc


def read_best_fit_table(output_prefix: Path) -> pd.DataFrame:
    fit_csv = output_prefix.with_name(output_prefix.name + "_fit_results.csv")
    return pd.read_csv(fit_csv)


def plot_instrument_spectrum(output_prefix: Path, instrument_name: str):
    csv_path = output_prefix.with_name(output_prefix.name + f"_{instrument_name}_plot.csv")
    df = pd.read_csv(csv_path)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(df["energy_keV"], df["rate"], xerr=df["energy_err"], yerr=df["rate_err"],
                fmt="o", ms=3, alpha=0.8, label=f"{instrument_name.upper()} data")
    ax.plot(df["energy_keV"], df["model_rate"], lw=2, label="Best-fit model")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Energy (keV)")
    ax.set_ylabel("Count rate")
    ax.set_title(f"{instrument_name.upper()} spectrum and model")
    ax.legend()
    plt.show()


In [ ]:
# validate_setup(setup)
xcm_path = build_three_spec_xcm(setup, output_prefix, redshift, nh_1e22)
print(f"Wrote XSPEC script: {xcm_path}")

# run_xspec(xcm_path)
# fit_df = read_best_fit_table(output_prefix)
# fit_df
# plot_instrument_spectrum(output_prefix, "m1")
# plot_instrument_spectrum(output_prefix, "m2")
# plot_instrument_spectrum(output_prefix, "pn")


## Notes

- This notebook does not rely on XGA internals.
- It assumes spectrum/background/RMF/ARF are already generated (e.g., with SAS).
- `nh_1e22` is in units of `10^22 cm^-2` for `tbabs`.
- Model: `constant*tbabs*apec` with shared physical parameters and free cross-calibration constants for M2/PN.
